# 01 — EDA и предобработка данных

Анализ эмоциональной тональности русскоязычных текстов.

**Этапы:**
1. Загрузка данных
2. Разведочный анализ (EDA)
3. Предобработка текстов
4. Сохранение обработанного датасета

In [ ]:
import sys, os

# Add project root to path when running on Kaggle
PROJECT_ROOT = "/kaggle/input/sentiment-analysis" if os.path.exists("/kaggle") else os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "../results"
os.makedirs(WORKING_DIR, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Working dir:  {WORKING_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('Libraries loaded')

## 1. Загрузка данных

In [ ]:
from src.data_loader import load_data

# Загружаем данные.
# Если есть CSV-файл — передайте путь: load_data(csv_path='/kaggle/input/.../data.csv')
# Иначе автоматически ищется датасет на HuggingFace или в /kaggle/input
dataset, info = load_data()

print('\nСтатистика по сплитам:')
for split, stat in info.items():
    print(f"  {split:12s}: {stat['size']:>6,} примеров | {stat['label_distribution']}")

In [ ]:
# Преобразуем в DataFrame для анализа
train_df = dataset['train'].to_pandas()
val_df   = dataset['validation'].to_pandas()
test_df  = dataset['test'].to_pandas()
all_df   = pd.concat([train_df, val_df, test_df], ignore_index=True)

print(f'Всего примеров: {len(all_df):,}')
print(f"Колонки: {all_df.columns.tolist()}")
all_df.head()

## 2. Разведочный анализ (EDA)

In [ ]:
LABEL_NAMES = {0: 'negative', 1: 'neutral', 2: 'positive'}
all_df['label_name'] = all_df['label'].map(LABEL_NAMES)

# Распределение классов
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# По всему датасету
counts = all_df['label_name'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#e74c3c', '#95a5a6', '#2ecc71'])
axes[0].set_title('Распределение классов (весь датасет)')
axes[0].set_xlabel('Класс')
axes[0].set_ylabel('Количество')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}\n({v/len(all_df)*100:.1f}%)', ha='center')

# По сплитам
split_counts = pd.DataFrame({
    split: df['label'].map(LABEL_NAMES).value_counts(normalize=True)
    for split, df in [('train', train_df), ('val', val_df), ('test', test_df)]
}).fillna(0)
split_counts.plot(kind='bar', ax=axes[1], color=['#3498db', '#e67e22', '#9b59b6'])
axes[1].set_title('Распределение по сплитам (доли)')
axes[1].set_xlabel('Класс')
axes[1].set_ylabel('Доля')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Длины текстов
all_df['text_len'] = all_df['text'].str.len()
all_df['word_count'] = all_df['text'].str.split().str.len()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col, title in zip(
    axes.flat,
    ['text_len', 'word_count', 'text_len', 'word_count'],
    ['Длина текста (символы)', 'Количество слов', 'Длина по классам', 'Слов по классам']
):
    if 'по классам' in title:
        for label_name, color in zip(['negative', 'neutral', 'positive'], ['#e74c3c', '#95a5a6', '#2ecc71']):
            subset = all_df[all_df['label_name'] == label_name][col]
            ax.hist(subset, bins=50, alpha=0.6, color=color, label=label_name, density=True)
        ax.legend()
    else:
        ax.hist(all_df[col], bins=50, color='#3498db', alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.set_ylabel('Частота')

plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(all_df[['text_len', 'word_count']].describe().round(1))

In [ ]:
# Примеры текстов по классам
for label_id, label_name in LABEL_NAMES.items():
    print(f'\n{"="*60}')
    print(f'  {label_name.upper()} (label={label_id})')
    print(f'{"="*60}')
    samples = all_df[all_df['label'] == label_id]['text'].sample(min(3, len(all_df[all_df['label'] == label_id])), random_state=42)
    for i, text in enumerate(samples, 1):
        print(f'{i}. {text[:200]}...' if len(text) > 200 else f'{i}. {text}')

## 3. Предобработка текстов

In [ ]:
from src.preprocessor import clean_text, preprocess_batch

# Проверяем очистку на примерах
test_texts = [
    'Отличный товар!!! Очень доволен 😊 #покупка http://example.com',
    '@user Ужасный сервис, больше не приду!!!',
    'Нормально, ничего особенного... https://shop.ru',
]

print('Примеры предобработки:')
for text in test_texts:
    cleaned = clean_text(text)
    print(f'\n  Исходный: {text}')
    print(f'  Очищенный: {cleaned}')

In [ ]:
# Применяем предобработку ко всему датасету
print('Применяю предобработку...')
dataset_clean = dataset.map(
    lambda batch: preprocess_batch(batch, text_column='text', clean=True, lemmatize_texts=False),
    batched=True,
    batch_size=1000,
    desc='Preprocessing',
)
print('Готово!')

# Проверяем результат
print('\nПервые 3 примера после очистки:')
for i in range(3):
    orig = dataset['train'][i]['text']
    clean_t = dataset_clean['train'][i]['text']
    print(f'  [{i}] {orig[:80]} → {clean_t[:80]}')

In [ ]:
# Проверяем, что предобработка не потеряла данные
for split in ['train', 'validation', 'test']:
    orig_size  = len(dataset[split])
    clean_size = len(dataset_clean[split])
    print(f'{split}: {orig_size} → {clean_size} примеров')

## 4. Сохранение обработанных данных

In [ ]:
# Сохраняем обработанный датасет
DATA_SAVE_PATH = f'{WORKING_DIR}/processed_data'
dataset_clean.save_to_disk(DATA_SAVE_PATH)
print(f'Датасет сохранён: {DATA_SAVE_PATH}')

# Также сохраняем в CSV для удобства
for split in ['train', 'validation', 'test']:
    df = dataset_clean[split].to_pandas()
    path = f'{WORKING_DIR}/{split}.csv'
    df.to_csv(path, index=False, encoding='utf-8')
    print(f'  {split}.csv: {len(df):,} строк → {path}')

In [ ]:
print('\n' + '='*60)
print('ИТОГИ ПРЕДОБРАБОТКИ')
print('='*60)
for split in ['train', 'validation', 'test']:
    df = dataset_clean[split].to_pandas()
    dist = df['label'].map(LABEL_NAMES).value_counts().to_dict()
    print(f'{split:12s}: {len(df):>6,} примеров | {dist}')
print(f'\nДанные сохранены в: {WORKING_DIR}')